In [8]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from scipy.optimize import curve_fit, minimize
import astropy.units as u
from astropy.io import fits
from astropy.table import Table, vstack, join
from scipy.stats import ttest_rel
from IPython.display import display, Math
import math
from scipy.odr import ODR, Model, RealData
from pathlib import Path

In [9]:
def red_chi_squared(y_values, y_uncertainties, model_data, dof):

  chi_squared = np.sum(((model_data - y_values) / y_uncertainties)**2)

  red_chi = chi_squared / (len(y_values) - dof)

  return red_chi

In [10]:
def open_csv_table(path):

    table = Table.read(path, format="csv")
    
    # print("Table headers: ", table.colnames)
    
    """
    stellar_fields = ["ID", "RA","Dec","UV1500Best","StellarMassBest","StellarMassQ50", "StellarMassQ16","StellarMassQ84", "SFR10Best","SFR100Best","ZfinalBest"]
    
    stellar_AGN_fields = ["ID", "RA","Dec","AGNanBest","AGNlumQ50","UV1500AGNBest","UV1500Best","StellarMassBest","StellarMassQ50", "StellarMassQ16","StellarMassQ84", "SFR10Best","SFR100Best","ZfinalBest"]
    
    if path == "/nvme/scratch/work/alberttg/Summer_project/photoZ_bc03_snorm_astro_cat.csv":
    # Data for stellar only fitting
    # Could probably automate this 
        new_table = table[stellar_fields]
        # Data from fits with just stellar
        
    if path == "/nvme/scratch/work/alberttg/Summer_project/photoZ_bc03_snorm_fritz_astro_cat.csv":
    # Stellar and AGN fit data   
        new_table = table[stellar_AGN_fields]
    
    """
    return table

In [11]:
def open_fits_table(file_path, ext):
        # Open the FITS file
        with fits.open(file_path) as hdul:
            # Show the HDU structure
            # hdul.info()
            
            data = hdul[ext].data

        # Convert to an Astropy Table
        tbl = Table(data)
        #print("Number of objects = ", len(tbl))
        return tbl

In [ ]:
def colour_SED_data(fit_model):

    Id = np.arange(1,145,1)
    #print(Id)

    galaxy_colour = np.empty(len(Id))
    g_c_lower_err = np.empty(len(Id))
    g_c_upper_err = np.empty(len(Id))

    for i in Id:

        path = f"/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/prospect_seds/{fit_model}_fits/galID_{i}.fits"
        path = Path(path)
        if not path.exists():
            print(f"{path} could not be found. Galaxy colour is thus replaced with nan.")
            galaxy_colour[i-1] = np.nan
            g_c_upper_err[i-1] = np.nan
            g_c_lower_err[i-1] = np.nan
            continue

        L_table = open_fits_table(path, 2)

        restwave = np.asarray(L_table["restWave"].data, dtype=float)

        lum_Q50 = np.asarray(L_table["lumQ50"].data, dtype=float)
        lum_Q16 = np.asarray(L_table["lumQ16"].data, dtype=float)
        lum_Q84 = np.asarray(L_table["lumQ84"].data, dtype=float)

        """
        def flux_nu(lum):
            lum_erg = lum * 3.828e33 # erg /s / A
            flux_10pc = lum_erg / (4 * np.pi * (10 * 3.0857e19)**2) # erg /s / cm^2 / A
            flux_nu = flux_10pc * 3.34e4 * 4000 ** 2 # Jy
            return flux_nu

        fig, ax = plt.subplots(figsize=(10,10))
        ax.plot(restwave, flux_nu(lum_Q50), color="black",alpha=1)
        ax.plot(restwave, flux_nu(lum_Q16), color="red",alpha=0.5)
        ax.plot(restwave, flux_nu(lum_Q84), color="blue",alpha=0.5)
        mask = (restwave <= 3000) & (restwave >= 6000)
        ax.set_xlim(np.min(restwave[mask]), np.max(restwave[mask]))
        ax.set_ylim(np.min(flux_nu(lum_Q16)[mask]), np.max(flux_nu(lum_Q84)[mask]))
        ax.set_ylabel(r"$flux_{\nu}$")
        ax.set_xlabel(r"Rest wavelength / $\AA$")
        plt.show()
        """
        #----------------------
        def calculate_mag_4200(lum_quartile):
            mask_4200 = (restwave >= 4150) & (restwave <= 4250) # Angstroms, tophat 100 A in width

            lum_4200 = np.mean(lum_quartile[mask_4200]) # L_solar / A

            lum_4200_erg = lum_4200 * 3.828e33 # erg /s / A

            flux_4200_10pc = lum_4200_erg / (4 * np.pi * (10 * 3.0857e19)**2) # erg /s / cm^2 / A

            flux_4200_nu = flux_4200_10pc * 3.34e4 * 4200 ** 2 # Jy

            M_abs_4200 = -2.5 * np.log10(flux_4200_nu) + 8.9

            return M_abs_4200

        M_abs_4200_Q50 = calculate_mag_4200(lum_Q50)
        M_abs_4200_Q16 = calculate_mag_4200(lum_Q16)
        M_abs_4200_Q84 = calculate_mag_4200(lum_Q84)
        M_abs_4200_upper_err = M_abs_4200_Q84 - M_abs_4200_Q50
        M_abs_4200_lower_err = M_abs_4200_Q50 - M_abs_4200_Q16

        #----------------------
        def calculate_mag_6000(lum_quartile):

            mask_6000 = (restwave >= 5950) & (restwave <= 6050) # Angstroms, tophat 100 A in width

            lum_6000 = np.mean(lum_quartile[mask_6000]) # L_solar / A

            lum_6000_erg = lum_6000 * 3.828e33 # erg /s / A

            flux_6000_10pc = lum_6000_erg / (4 * np.pi * (10 * 3.0857e19)**2) # erg /s / cm^2 / A

            flux_6000_nu = flux_6000_10pc * 3.34e4 * 6000 ** 2 # Jy

            M_abs_6000 = -2.5 * np.log10(flux_6000_nu) + 8.9

            return M_abs_6000
        #----------------------
        M_abs_6000_Q50 = calculate_mag_6000(lum_Q50)
        M_abs_6000_Q16 = calculate_mag_6000(lum_Q16)
        M_abs_6000_Q84 = calculate_mag_6000(lum_Q84)
        M_abs_6000_upper_err = M_abs_6000_Q84 - M_abs_6000_Q50
        M_abs_6000_lower_err = M_abs_6000_Q50 - M_abs_6000_Q16
        
        galaxy_colour[i-1] = M_abs_4200_Q50 - M_abs_6000_Q50
        g_c_upper_err[i-1] = np.sqrt( (M_abs_4200_upper_err)**2 + (M_abs_6000_upper_err)**2)
        g_c_lower_err[i-1] = np.sqrt( (M_abs_4200_lower_err)**2 + (M_abs_6000_lower_err)**2)
    
    if fit_model == "photoZ_bc03_snorm":
        csv_path = "/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/bc03_snorm_astro_cat.csv"

    if fit_model == "photoZ_bc03_snorm_fritz_lbol_prior":
        csv_path = "/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/bc03_snorm_fritz_lbol_prior_astro_cat.csv"

    table = open_csv_table(csv_path)
    table["galaxy_colour"] = galaxy_colour
    table["galaxy_colour_upper_err"] = g_c_upper_err
    table["galaxy_colour_lower_err"] = g_c_lower_err

    table.write(csv_path, format="csv", overwrite=True)
    
    return galaxy_colour, g_c_upper_err, g_c_lower_err

In [13]:
def uv_slope_SED_data(fit_model):

    Id = np.arange(1,145,1)

    uv_slope = np.empty(len(Id))
    uv_slope_err = np.empty(len(Id))

    # Calzetti et al. (1994) UV continuum windows, rest-frame Angstroms
    CALZETTI_WINDOWS = [
        (1268, 1284),
        (1309, 1316),
        (1342, 1371),
        (1407, 1515),
        (1562, 1583),
        (1677, 1740),
        (1760, 1833),
        (1866, 1890),
        (1930, 1950),
        (2400, 2580),
    ]

    def linear_model(B, x):
        return B[0] * x + B[1]

    def calculate_beta_odr(lum_Q50, lum_Q16, lum_Q84, restwave,
                            windows=CALZETTI_WINDOWS, min_windows=2):
        """
        Fits f_lambda ~ lambda**beta over the Calzetti+94 windows using ODR,
        with per-window flux uncertainties (from Q16/Q84) propagated into
        log-space y-errors. Returns (beta, beta_err), or (nan, nan) if too
        few windows are usable.
        """
        centers = []
        fluxes = []
        flux_errs = []

        for (lo, hi) in windows:
            mask = (restwave >= lo) & (restwave <= hi)
            if not np.any(mask):
                continue

            f50 = np.mean(lum_Q50[mask]) # the beta slope is the same for log_flux vs log_wavelength as for log_L vs log_wavelength
            f16 = np.mean(lum_Q16[mask])
            f84 = np.mean(lum_Q84[mask])

            centers.append((lo + hi) / 2.0)
            fluxes.append(f50)
            # symmetrised 1-sigma flux error per window
            flux_errs.append(0.5 * (f84 - f16))

        centers = np.asarray(centers, dtype=float)
        fluxes = np.asarray(fluxes, dtype=float)
        flux_errs = np.asarray(flux_errs, dtype=float)

        valid = np.isfinite(fluxes) & (fluxes > 0) & np.isfinite(flux_errs) & (flux_errs > 0)

        if np.sum(valid) < min_windows:
            return np.nan, np.nan

        log_x = np.log10(centers[valid])
        log_y = np.log10(fluxes[valid])
        # propagate linear-space flux error into log-space error:
        # d(log10 f)/df = 1 / (f * ln10)
        log_y_err = flux_errs[valid] / (fluxes[valid] * np.log(10))

        data = RealData(log_x, log_y, sy=log_y_err)
        model = Model(linear_model)

        # initial guess from an unweighted polyfit, to help ODR converge
        p0 = np.polyfit(log_x, log_y, 1)

        odr = ODR(data, model, beta0=p0)
        output = odr.run()

        beta = output.beta[0]
        beta_err = output.sd_beta[0]

        return beta, beta_err
        """
        results = fit_linear_mcmc(x_in=log_x, xerr=None, y_in=log_y, yerr=log_y_err, x_pivot=None, n_walkers=50, n_steps=2000, plot=False, save_path=None)

        gradient = results["m_median"]
        gradient_upper_err = results["m_84"] - results["m_median"]
        gradient_lower_err = results["m_median"] - results["m_16"]
        intercept = results["c_median"]
        int_scatter_med = results["sig_int_median"]

        print(f"Red chi squared = {red_chi_squared(y_values=log_y, y_uncertainties=log_y_err, model_data=(log_x * gradient + intercept), dof=2):.3f})"
        """
    for i in Id:

        path = f"/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/prospect_seds/{fit_model}_fits/galID_{i}.fits"
        path = Path(path)
        if not path.exists():
            print(f"{path} could not be found. UV slope is thus replaced with nan.")
            uv_slope[i-1] = np.nan
            uv_slope_err[i-1] = np.nan
            continue

        L_table = open_fits_table(path, 2)

        restwave = np.asarray(L_table["restWave"].data, dtype=float)

        lum_Q50 = np.asarray(L_table["lumQ50"].data, dtype=float)
        lum_Q16 = np.asarray(L_table["lumQ16"].data, dtype=float)
        lum_Q84 = np.asarray(L_table["lumQ84"].data, dtype=float)

        beta, beta_err = calculate_beta_odr(lum_Q50, lum_Q16, lum_Q84, restwave)

        uv_slope[i-1] = beta
        uv_slope_err[i-1] = beta_err

    if fit_model == "photoZ_bc03_snorm":
        csv_path = "/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/bc03_snorm_astro_cat.csv"

    if fit_model == "photoZ_bc03_snorm_fritz_lbol_prior":
        csv_path = "/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/bc03_snorm_fritz_lbol_prior_astro_cat.csv"

    table = open_csv_table(csv_path)
    table["uv_slope"] = uv_slope
    table["uv_slope_err"] = uv_slope_err

    table.write(csv_path, format="csv", overwrite=True)

    return uv_slope, uv_slope_err

In [ ]:
colour_SED_data("photoZ_bc03_snorm_fritz_lbol_prior")
uv_slope_SED_data("photoZ_bc03_snorm_fritz_lbol_prior")

colour_SED_data("photoZ_bc03_snorm")
uv_slope_SED_data("photoZ_bc03_snorm")

/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/prospect_seds/photoZ_bc03_snorm_fritz_lbol_prior_fits/galID_110.fits could not be found. Galaxy colour is thus replaced with nan.
/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/prospect_seds/photoZ_bc03_snorm_fritz_lbol_prior_fits/galID_130.fits could not be found. Galaxy colour is thus replaced with nan.
/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/prospect_seds/photoZ_bc03_snorm_fritz_lbol_prior_fits/galID_110.fits could not be found. UV slope is thus replaced with nan.
/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/prospect_seds/photoZ_bc03_snorm_fritz_lbol_prior_fits/galID_130.fits could not be found. UV slope is thus replaced with nan.
/nvme/scratch/work/alberttg/Summer_project/Data_inputs/ProSpect_data/prospect_seds/photoZ_bc03_snorm_fits/galID_110.fits could not be found. Galaxy colour is thus replaced with nan.
/nvme/scratch/work/alberttg/Summ

(array([-1.89520982,  1.90252581,  0.96851819, -2.21567109, -2.00615746,
        -2.0477832 , -2.16537336,  4.71552063, -1.67163272, -2.35882136,
        -1.2142554 , -1.72037783, -1.81046195, -2.26687704, -1.88583584,
        -2.13863901, -1.44695954,  1.21707768,  0.40544533, -2.50421951,
        -1.87741654, -2.21336414, -1.91362778, -2.11909748, -2.38935295,
        -2.05257976, -1.30317485,  2.0381686 ,  1.72216976, -2.36102128,
        -1.09380992, -1.2839994 , -2.32591745, -1.71567307, -2.50081101,
        -1.81273255, -1.20118945, -1.95027403,  1.17768004, -2.21534843,
        -2.04738321,  2.73944314, -1.36795852,  4.96775127, -1.43378882,
        -1.71466196,  2.8935677 , -2.0542042 , -1.83589538, -1.34730533,
        -1.37743599, -1.79791223, -1.14389839, -2.28156206,  2.07640853,
        -2.00795641, -0.589863  , -0.83901772,  3.88265195, -2.24499289,
        -2.05064359,  3.28651826, -0.78549958, -1.66987132, -1.90759447,
        -0.97682782,  5.34644755,  4.41987354,  3.1